# 📊 Evaluación y Métricas de Machine Learning

## Objetivos
- Implementar Cross-Validation desde cero
- Métricas completas de evaluación
- ROC y AUC para clasificación
- Precision-Recall curves
- Grid Search manual
- Comparación de múltiples modelos

---

## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Conceptos Principales](#2)
- [3 - Ejercicios Prácticos](#3)
- [4 - Resumen](#4)

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
from pathlib import Path

# Agregar el directorio raíz al path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from utils.plot_utils import plot_confusion_matrix, plot_learning_curve
    print("✅ Entorno configurado correctamente")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("💡 Ejecuta: pip install -e . desde la raíz")
    raise

## 1. K-Fold Cross-Validation desde CERO

Cross-validation es esencial para evaluar modelos de manera robusta.

### ¿Por qué Cross-Validation?
- Usa todos los datos para entrenamiento y evaluación
- Reduce varianza en la estimación del rendimiento
- Detecta mejor el overfitting

### K-Fold CV:
1. Dividir datos en K partes (folds)
2. Para cada fold:
   - Entrenar con K-1 folds
   - Evaluar con 1 fold
3. Promediar los resultados

In [ ]:
class KFoldCV:
    """
    K-Fold Cross-Validation implementado desde cero.
    """
    
    def __init__(self, n_splits=5, shuffle=True, random_state=None):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
    
    def split(self, X, y=None):
        """
        Genera índices de train/test para cada fold.
        
        Yields:
        -------
        train_idx, test_idx : arrays de índices
        """
        n_samples = len(X)
        indices = np.arange(n_samples)
        
        if self.shuffle:
            if self.random_state is not None:
                np.random.seed(self.random_state)
            np.random.shuffle(indices)
        
        fold_sizes = np.full(self.n_splits, n_samples // self.n_splits, dtype=int)
        fold_sizes[:n_samples % self.n_splits] += 1
        
        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_idx = indices[start:stop]
            train_idx = np.concatenate([indices[:start], indices[stop:]])
            yield train_idx, test_idx
            current = stop

# Ejemplo de uso
X = np.arange(20).reshape(-1, 1)
y = np.arange(20)

kf = KFoldCV(n_splits=5, shuffle=True, random_state=42)

print("Ejemplo de K-Fold CV (K=5):")
for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    print(f"Fold {fold}:")
    print(f"  Train: {len(train_idx)} samples")
    print(f"  Test:  {len(test_idx)} samples")
    print(f"  Test indices: {test_idx}")

### Función para Evaluar con CV

In [ ]:
def cross_val_score(model, X, y, cv=5, scoring='accuracy'):
    """
    Evalúa un modelo usando K-Fold Cross-Validation.
    
    Parameters:
    -----------
    model : objeto con métodos fit() y predict()
    X : array-like
        Features
    y : array-like
        Target
    cv : int
        Número de folds
    scoring : str
        Métrica: 'accuracy', 'mse', 'r2'
    
    Returns:
    --------
    scores : list
        Scores de cada fold
    """
    kfold = KFoldCV(n_splits=cv, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, test_idx in kfold.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Entrenar
        model.fit(X_train, y_train)
        
        # Predecir
        y_pred = model.predict(X_test)
        
        # Calcular score según métrica
        if scoring == 'accuracy':
            score = np.mean(y_pred == y_test)
        elif scoring == 'mse':
            score = -np.mean((y_pred - y_test) ** 2)  # Negativo para que mayor sea mejor
        elif scoring == 'r2':
            ss_res = np.sum((y_test - y_pred) ** 2)
            ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
            score = 1 - (ss_res / ss_tot)
        else:
            raise ValueError(f"Scoring '{scoring}' no reconocido")
        
        scores.append(score)
    
    return np.array(scores)

# Ejemplo con modelo dummy
class ModeloDummy:
    def fit(self, X, y):
        self.mean = np.mean(y)
        return self
    
    def predict(self, X):
        return np.round(np.random.rand(len(X)))

# Generar datos sintéticos
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=100, n_features=5, random_state=42)

modelo = ModeloDummy()
scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')

print(f"Scores de cada fold: {scores}")
print(f"Media: {scores.mean():.4f}")
print(f"Desviación estándar: {scores.std():.4f}")

## 2. Métricas de Clasificación Completas

Implementación completa de todas las métricas importantes.

In [ ]:
class MetricasClasificacion:
    """
    Clase para calcular todas las métricas de clasificación binaria.
    """
    
    def __init__(self, y_true, y_pred, y_proba=None):
        self.y_true = np.array(y_true)
        self.y_pred = np.array(y_pred)
        self.y_proba = np.array(y_proba) if y_proba is not None else None
        
        # Calcular matriz de confusión
        self.TP = np.sum((self.y_true == 1) & (self.y_pred == 1))
        self.TN = np.sum((self.y_true == 0) & (self.y_pred == 0))
        self.FP = np.sum((self.y_true == 0) & (self.y_pred == 1))
        self.FN = np.sum((self.y_true == 1) & (self.y_pred == 0))
    
    def accuracy(self):
        return (self.TP + self.TN) / (self.TP + self.TN + self.FP + self.FN)
    
    def precision(self):
        return self.TP / (self.TP + self.FP) if (self.TP + self.FP) > 0 else 0
    
    def recall(self):
        return self.TP / (self.TP + self.FN) if (self.TP + self.FN) > 0 else 0
    
    def f1_score(self):
        p = self.precision()
        r = self.recall()
        return 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    
    def specificity(self):
        return self.TN / (self.TN + self.FP) if (self.TN + self.FP) > 0 else 0
    
    def confusion_matrix(self):
        return np.array([[self.TN, self.FP], [self.FN, self.TP]])
    
    def classification_report(self):
        print("="*50)
        print("REPORTE DE CLASIFICACIÓN")
        print("="*50)
        print(f"Accuracy:     {self.accuracy():.4f}")
        print(f"Precision:    {self.precision():.4f}")
        print(f"Recall:       {self.recall():.4f}")
        print(f"F1-Score:     {self.f1_score():.4f}")
        print(f"Specificity:  {self.specificity():.4f}")
        print("\nMatriz de Confusión:")
        print(self.confusion_matrix())
        print("="*50)

# Ejemplo
y_true = np.array([0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1])
y_pred = np.array([0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0])

metricas = MetricasClasificacion(y_true, y_pred)
metricas.classification_report()

## 3. ROC Curve y AUC

La curva ROC (Receiver Operating Characteristic) muestra el trade-off entre True Positive Rate y False Positive Rate.

In [ ]:
def roc_curve(y_true, y_proba):
    """
    Calcula la curva ROC.
    
    Returns:
    --------
    fpr, tpr, thresholds
    """
    # Ordenar por probabilidad descendente
    desc_score_indices = np.argsort(y_proba)[::-1]
    y_proba = y_proba[desc_score_indices]
    y_true = y_true[desc_score_indices]
    
    # Calcular TPR y FPR para cada threshold
    tpr_list = [0]
    fpr_list = [0]
    
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    
    tp = 0
    fp = 0
    
    for i in range(len(y_true)):
        if y_true[i] == 1:
            tp += 1
        else:
            fp += 1
        
        tpr = tp / n_pos if n_pos > 0 else 0
        fpr = fp / n_neg if n_neg > 0 else 0
        
        tpr_list.append(tpr)
        fpr_list.append(fpr)
    
    return np.array(fpr_list), np.array(tpr_list)

def auc_score(y_true, y_proba):
    """
    Calcula el área bajo la curva ROC usando la regla del trapecio.
    """
    fpr, tpr = roc_curve(y_true, y_proba)
    # Usar regla del trapecio
    return np.trapz(tpr, fpr)

# Ejemplo
np.random.seed(42)
y_true = np.array([0, 0, 1, 1, 0, 1, 1, 0, 1, 0])
y_proba = np.array([0.1, 0.4, 0.35, 0.8, 0.2, 0.9, 0.7, 0.3, 0.85, 0.15])

fpr, tpr = roc_curve(y_true, y_proba)
auc = auc_score(y_true, y_proba)

# Visualizar
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC Score: {auc:.4f}")

## 4. Precision-Recall Curve

Útil cuando las clases están desbalanceadas.

In [ ]:
def precision_recall_curve(y_true, y_proba):
    """
    Calcula la curva Precision-Recall.
    """
    # Ordenar por probabilidad descendente
    desc_indices = np.argsort(y_proba)[::-1]
    y_proba = y_proba[desc_indices]
    y_true = y_true[desc_indices]
    
    precision_list = []
    recall_list = []
    
    n_pos = np.sum(y_true == 1)
    tp = 0
    
    for i in range(len(y_true)):
        if y_true[i] == 1:
            tp += 1
        
        precision = tp / (i + 1)
        recall = tp / n_pos if n_pos > 0 else 0
        
        precision_list.append(precision)
        recall_list.append(recall)
    
    return np.array(precision_list), np.array(recall_list)

# Ejemplo
precision, recall = precision_recall_curve(y_true, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Grid Search Manual

Búsqueda exhaustiva de hiperparámetros.

In [ ]:
class GridSearchCV:
    """
    Grid Search con Cross-Validation implementado desde cero.
    """
    
    def __init__(self, model_class, param_grid, cv=5, scoring='accuracy'):
        self.model_class = model_class
        self.param_grid = param_grid
        self.cv = cv
        self.scoring = scoring
        self.results = []
        self.best_params = None
        self.best_score = -np.inf
    
    def fit(self, X, y):
        # Generar todas las combinaciones de parámetros
        param_combinations = self._get_param_combinations()
        
        print(f"Probando {len(param_combinations)} combinaciones de parámetros...\n")
        
        for params in param_combinations:
            # Crear modelo con estos parámetros
            model = self.model_class(**params)
            
            # Evaluar con cross-validation
            scores = cross_val_score(model, X, y, cv=self.cv, scoring=self.scoring)
            mean_score = scores.mean()
            std_score = scores.std()
            
            self.results.append({
                'params': params,
                'mean_score': mean_score,
                'std_score': std_score,
                'scores': scores
            })
            
            print(f"Params: {params}")
            print(f"  Score: {mean_score:.4f} (+/- {std_score:.4f})")
            
            # Actualizar mejor configuración
            if mean_score > self.best_score:
                self.best_score = mean_score
                self.best_params = params
        
        print(f"\n🏆 Mejor configuración:")
        print(f"   Params: {self.best_params}")
        print(f"   Score: {self.best_score:.4f}")
        
        return self
    
    def _get_param_combinations(self):
        """Genera todas las combinaciones de parámetros."""
        keys = list(self.param_grid.keys())
        values = list(self.param_grid.values())
        
        combinations = []
        self._generate_combinations(keys, values, 0, {}, combinations)
        return combinations
    
    def _generate_combinations(self, keys, values, index, current, combinations):
        """Recursivamente genera combinaciones."""
        if index == len(keys):
            combinations.append(current.copy())
            return
        
        for value in values[index]:
            current[keys[index]] = value
            self._generate_combinations(keys, values, index + 1, current, combinations)

# Ejemplo con modelo dummy
class ModeloConParams:
    def __init__(self, param1=1, param2=0.1):
        self.param1 = param1
        self.param2 = param2
    
    def fit(self, X, y):
        return self
    
    def predict(self, X):
        # Dummy prediction que depende de los parámetros
        return np.random.rand(len(X)) > (0.5 - self.param2)

# Grid de parámetros a probar
param_grid = {
    'param1': [1, 2, 3],
    'param2': [0.1, 0.2]
}

# Ejecutar Grid Search
grid_search = GridSearchCV(ModeloConParams, param_grid, cv=3, scoring='accuracy')
grid_search.fit(X, y)

## 🎓 Resumen

Has aprendido:

1. ✅ K-Fold Cross-Validation desde cero
2. ✅ Métricas completas de clasificación
3. ✅ Curva ROC y AUC
4. ✅ Curva Precision-Recall
5. ✅ Grid Search manual

Estas herramientas son esenciales para evaluar y optimizar modelos de ML de manera profesional.

---

## 📝 Ejercicio Final

Usa todas las herramientas aprendidas para evaluar un modelo real de clasificación.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Genera un dataset de clasificación
# 2. Implementa un modelo simple (puedes usar el del notebook de regresión logística)
# 3. Evalúa con K-Fold CV
# 4. Calcula todas las métricas
# 5. Grafica ROC curve
# 6. Encuentra los mejores hiperparámetros con Grid Search

print("¡Implementa tu evaluación completa aquí!")